In [ ]:
import os
import mne
import numpy as np
import pandas as pd
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import matplotlib


# Step 1: Read labels

label_file = r"E:\TDBRAIN_participants_V2.xlsx"
df = pd.read_excel(label_file)
df = df[df['formal_status'].notna()]

def get_label(formal_status):
    if isinstance(formal_status, str):
        if formal_status.upper() == 'MDD':
            return 1
        elif formal_status.upper() == 'HEALTHY':
            return 0
    return None

df['label'] = df['formal_status'].apply(get_label)
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)
print("Label distribution:")
print(df['label'].value_counts())

target_dict = dict(zip(df['participants_ID'], df['label']))

# Step 2: Read EEG data and extract features

data_dir = r"E:\preprocessed_data_alpha"

# Store features and labels
all_features = []
all_labels = []

# Iterate each subject in the label table, check ses-1 ~ ses-4, and use only the first existing session
for idx, row in df.iterrows():
    pid = row["participants_ID"]
    label = row["label"]
    subject_dir = os.path.join(data_dir, pid)
    if not os.path.isdir(subject_dir):
        print(f"Warning: subject folder {pid} not found!")
        continue
    for ses in range(1, 5):
        fif_path = os.path.join(subject_dir, f"ses-{ses}", "eeg", "preprocessed-epo.fif")
        if os.path.exists(fif_path):
            try:
                epochs = mne.read_epochs(fif_path, preload=True, verbose=False)
                data = epochs.get_data()  # shape: (n_epochs, n_channels, n_times)
                features = data.reshape(data.shape[0], -1)  # shape: (n_epochs, n_channels*n_times)
                all_features.append(features)
                all_labels += [label] * features.shape[0]
                break  
            except Exception as e:
                print(f"Error reading {fif_path}: {e}")
    else:
        print(f"Warning: for subject {pid}, no valid session data was found")


X = np.concatenate(all_features, axis=0)
y = np.array(all_labels)


# ========== Step 3: t-SNE ==========
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_scaled)

print("Original feature dimension:", X.shape[1])      # high dimension
print("t-SNE output dimension:", X_tsne.shape[1])     # 2D

# ========== Step 4: visualization ==========
plt.figure(figsize=(10, 7))
plt.title("HC vs MDD EEG Data")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")


plt.scatter(X_tsne[y==0, 0], X_tsne[y==0, 1], c='green', label='HC', alpha=0.5)
plt.scatter(X_tsne[y==1, 0], X_tsne[y==1, 1], c='red', label='MDD', alpha=0.5)

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import glob, os
import numpy as np

base_dir = r'E:\225A'  # path
smooth_window = 7      
max_epoch = 200        

# collect file
shallow_files = glob.glob(os.path.join(base_dir, 'training_shallow *.csv'))
m_files       = glob.glob(os.path.join(base_dir, 'training_mshallow *.csv'))

# load data
shallow_data = {}
for fp in shallow_files:
    df = pd.read_csv(fp)
    if 'epoch' not in df.columns: df['epoch'] = df.index + 1
    band = os.path.splitext(os.path.basename(fp))[0].split('shallow',1)[1].strip(' _')
    if all(col in df.columns for col in ['sensitivity','specificity']):
        df['balanced_acc'] = (df['sensitivity'] + df['specificity'])/2
    shallow_data[band] = df

m_data = {}
for fp in m_files:
    df = pd.read_csv(fp)
    if 'epoch' not in df.columns: df['epoch'] = df.index + 1
    band = os.path.splitext(os.path.basename(fp))[0].split('mshallow',1)[1].strip(' _')
    if all(col in df.columns for col in ['sensitivity','specificity']):
        df['balanced_acc'] = (df['sensitivity'] + df['specificity'])/2
    m_data[band] = df

# same bands
bands = sorted(set(shallow_data.keys()) & set(m_data.keys()))

# standards
metrics = ['sensitivity', 'specificity', 'f1_score', 'auc', 'balanced_acc']
metric_names = {
    'sensitivity': 'Sensitivity',
    'specificity': 'Specificity',
    'f1_score': 'F1 Score',
    'auc': 'AUC',
    'balanced_acc': 'Balanced Accuracy'
}

# plot
for metric in metrics:
    shallow_vals, m_vals, valid_bands = [], [], []
    for band in bands:
        vs = shallow_data[band].loc[shallow_data[band]['epoch']==max_epoch, metric]
        vm = m_data[band].loc[m_data[band]['epoch']==max_epoch, metric]
        if not vs.empty and not vm.empty:
            shallow_vals.append(vs.values[0])
            m_vals.append(vm.values[0])
            valid_bands.append(band)
    if not valid_bands: continue

    x = np.arange(len(valid_bands))
    width = 0.35
    plt.figure(figsize=(8,5))
    plt.bar(x - width/2, shallow_vals, width, label='ShallowConvNet')
    plt.bar(x + width/2, m_vals, width, label='M-ShallowConvNet')
    plt.xticks(x, valid_bands, rotation=0)  
    plt.ylabel(metric_names[metric])
    plt.title(f'{metric_names[metric]} (Epoch={max_epoch})')
    plt.ylim(max(0, min(shallow_vals+m_vals)*0.95), min(1, max(shallow_vals+m_vals)*1.05))
    for i in x:
        plt.text(i - width/2, shallow_vals[i] + 0.005, f'{shallow_vals[i]:.3f}', ha='center')
        plt.text(i + width/2, m_vals[i] + 0.005, f'{m_vals[i]:.3f}', ha='center')
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

#  ShallowConvNet--Loss and Accuracy curve
plt.figure(figsize=(10, 6))
for band in bands:
    df = shallow_data[band]
    loss_smooth = df['train_loss'].rolling(window=smooth_window, min_periods=1).mean()
    plt.plot(df['epoch'], loss_smooth, label=band)
plt.title('ShallowConvNet Loss Curve')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

plt.figure(figsize=(10, 6))
for band in bands:
    df = shallow_data[band]
    acc_smooth = df['val_acc'].rolling(window=smooth_window, min_periods=1).mean()
    plt.plot(df['epoch'], acc_smooth, label=band)
plt.title('ShallowConvNet Accuracy Curve')
plt.xlabel('Epoch'); plt.ylabel('Accuracy')
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

# M-ShallowConvNet--Loss and Accuracy curve
plt.figure(figsize=(10, 6))
for band in bands:
    df = m_data[band]
    loss_smooth = df['train_loss'].rolling(window=smooth_window, min_periods=1).mean()
    plt.plot(df['epoch'], loss_smooth, label=band)
plt.title('M-ShallowConvNet Loss Curve')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

plt.figure(figsize=(10, 6))
for band in bands:
    df = m_data[band]
    acc_smooth = df['val_acc'].rolling(window=smooth_window, min_periods=1).mean()
    plt.plot(df['epoch'], acc_smooth, label=band)
plt.title('M-ShallowConvNet Accuracy Curve')
plt.xlabel('Epoch'); plt.ylabel('Accuracy')
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

